# OMNISIGNAL
### Multi-Omics Discordance & Hidden Molecular Subtype Discovery Engine

A computational biology project that combines **gene expression, DNA methylation and clinical measurements** to discover hidden molecular subgroups and detect samples whose biological data layers disagree.

**Research question:** Can integrating multiple biological measurement layers reveal patterns that single-omics analysis misses?

> This notebook uses a reproducible synthetic cohort. Results demonstrate the workflow only; they are not biological discoveries or clinical evidence.

## What this project delivers
1. Multi-omics data generation and quality audit.
2. Independent preprocessing of each modality.
3. Single-omics versus integrated clustering.
4. Cross-modal discordance and anomaly detection.
5. Feature signatures and exploratory outcome modelling.
6. Robustness checks and exportable research results.

**Distinctive angle:** It studies not only molecular similarity, but also *discordance between biological layers*.

## 1. Setup

In [ ]:
import os, json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, classification_report, roc_auc_score
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split

SEED=42
rng=np.random.default_rng(SEED)
np.random.seed(SEED)
OUT="omnisignal_outputs"
os.makedirs(OUT, exist_ok=True)
print("OMNISIGNAL ready")

## 2. Generate a synthetic multi-omics cohort
We simulate three latent subtypes with distinct expression and methylation signatures. A small subset receives an artificial shift in one modality to test discordance detection.

In [ ]:
n, ng, nm = 600, 30, 20
truth=rng.choice(3,n,p=[.38,.34,.28])
ec=rng.normal(0,1,(3,ng)); mc=rng.normal(0,1,(3,nm))
ec[0,:8]+=2; ec[1,8:16]+=2; ec[2,16:24]+=2
mc[0,:6]-=1.8; mc[1,6:13]+=1.8; mc[2,13:]-=1.8
expr=ec[truth]+rng.normal(0,.85,(n,ng))
meth=mc[truth]+rng.normal(0,.85,(n,nm))
clin=pd.DataFrame({
 "age":rng.normal(54+4*truth,11,n).clip(18,90),
 "inflammation_score":rng.normal(40+13*truth,12,n).clip(0,100),
 "metabolic_score":rng.normal(35+15*(truth==2),13,n).clip(0,100),
 "functional_score":rng.normal(85-12*truth,12,n).clip(0,100)
})
logit=-2+.65*truth+.025*(clin.inflammation_score-40)
clin["simulated_outcome"]=rng.binomial(1,1/(1+np.exp(-logit)))
expr_cols=[f"GENE_{i:02d}" for i in range(1,ng+1)]
meth_cols=[f"CG_SITE_{i:02d}" for i in range(1,nm+1)]
expr=pd.DataFrame(expr,columns=expr_cols)
meth=pd.DataFrame(meth,columns=meth_cols)
discord_idx=rng.choice(n,35,replace=False)
expr.iloc[discord_idx,:]+=rng.normal(0,2.4,(len(discord_idx),ng))
for frame in (expr,meth,clin):
    frame.mask(rng.random(frame.shape)<.025,inplace=True)
meta=pd.DataFrame({"sample_id":[f"OS_{i:04d}" for i in range(n)],
                   "simulation_truth":truth,
                   "injected_discordance":np.isin(np.arange(n),discord_idx)})
print(expr.shape,meth.shape,clin.shape)

## 3. Data quality audit

In [ ]:
def audit(name,df):
    return {"modality":name,"samples":len(df),"features":df.shape[1],
            "missing_cells":int(df.isna().sum().sum()),
            "missing_percent":round(df.isna().mean().mean()*100,2),
            "duplicate_rows":int(df.duplicated().sum())}
audit_df=pd.DataFrame([audit("Expression",expr),audit("Methylation",meth),audit("Clinical",clin)])
display(audit_df)

## 4. Preprocess each modality
Median imputation and standardisation are used as a transparent baseline. Real studies need assay-specific normalisation, batch-effect assessment and covariate harmonisation.

In [ ]:
def prep(df):
    imp=SimpleImputer(strategy="median")
    sc=StandardScaler()
    z=sc.fit_transform(imp.fit_transform(df))
    return pd.DataFrame(z,columns=df.columns)
E=prep(expr); METH=prep(meth); C=prep(clin.drop(columns="simulated_outcome"))
def block(df): return df/np.sqrt(df.shape[1])
integrated=pd.concat([block(E).add_prefix("EXP_"),
                      block(METH).add_prefix("METH_"),
                      block(C).add_prefix("CLIN_")],axis=1)
print("Integrated matrix:",integrated.shape)

## 5. Compare single-omics and integrated clustering

In [ ]:
datasets={"Expression only":E,"Methylation only":METH,"Clinical only":C,"Integrated":integrated}
labels={}; models={}; rows=[]
for name,X in datasets.items():
    km=KMeans(n_clusters=3,n_init=20,random_state=SEED)
    lab=km.fit_predict(X)
    models[name]=km; labels[name]=lab
    rows.append({"representation":name,"silhouette":silhouette_score(X,lab),
                 "ARI_vs_synthetic_truth":adjusted_rand_score(truth,lab)})
cluster_metrics=pd.DataFrame(rows).sort_values("ARI_vs_synthetic_truth",ascending=False)
display(cluster_metrics)

## 6. Visualise the integrated molecular landscape

In [ ]:
xy=PCA(n_components=2,random_state=SEED).fit_transform(integrated)
plt.figure(figsize=(9,6))
for g in np.unique(labels["Integrated"]):
    ix=labels["Integrated"]==g
    plt.scatter(xy[ix,0],xy[ix,1],s=22,alpha=.65,label=f"Cluster {g}")
plt.title("OMNISIGNAL integrated multi-omics landscape")
plt.xlabel("Principal component 1"); plt.ylabel("Principal component 2")
plt.legend(); plt.tight_layout(); plt.show()

## 7. Cross-modal discordance detection
Isolation Forest scores each modality independently. A large gap between a sample's modality scores flags a candidate discordant profile—not a diagnosis or proof of a biological mechanism.

In [ ]:
def anomaly_score(X):
    model=IsolationForest(n_estimators=250,contamination=.06,random_state=SEED)
    model.fit(X)
    return -model.decision_function(X)
anom=pd.DataFrame({"expression_anomaly":anomaly_score(E),
                   "methylation_anomaly":anomaly_score(METH),
                   "clinical_anomaly":anomaly_score(C)})
z=(anom-anom.mean())/anom.std(ddof=0)
anom["discordance_index"]=z.max(axis=1)-z.min(axis=1)
anom["dominant_modality"]=z.idxmax(axis=1).str.replace("_anomaly","",regex=False)
anom["sample_id"]=meta.sample_id
display(anom.sort_values("discordance_index",ascending=False).head(15))

### Plot the strongest discordance candidates

In [ ]:
top=anom.nlargest(20,"discordance_index").sort_values("discordance_index")
plt.figure(figsize=(10,5))
plt.bar(top.sample_id,top.discordance_index)
plt.xticks(rotation=75); plt.ylabel("Discordance index")
plt.title("Top 20 candidate cross-modal discordant samples")
plt.tight_layout(); plt.show()

## 8. Molecular signatures across clusters

In [ ]:
signature=integrated.copy()
signature["cluster"]=labels["Integrated"]
means=signature.groupby("cluster").mean()
display(means.iloc[:,:12].round(2))
plt.figure(figsize=(12,5))
plt.imshow(means.to_numpy(),aspect="auto")
plt.colorbar(label="Mean standardised feature")
plt.yticks(range(len(means)),[f"Cluster {x}" for x in means.index])
plt.xticks(range(len(means.columns)),means.columns,rotation=90,fontsize=7)
plt.title("Integrated cluster feature signatures")
plt.tight_layout(); plt.show()

## 9. Exploratory outcome modelling
The target is simulated. This section demonstrates a held-out evaluation workflow and must not be interpreted as clinical prediction.

In [ ]:
y=clin["simulated_outcome"].astype(int)
Xtr,Xte,ytr,yte=train_test_split(integrated,y,test_size=.25,random_state=SEED,stratify=y)
rf=RandomForestClassifier(n_estimators=300,min_samples_leaf=3,class_weight="balanced",random_state=SEED)
rf.fit(Xtr,ytr)
pred=rf.predict(Xte); prob=rf.predict_proba(Xte)[:,1]
print(classification_report(yte,pred,zero_division=0))
print("ROC-AUC:",round(roc_auc_score(yte,prob),3))
imp=pd.Series(rf.feature_importances_,index=integrated.columns).nlargest(15).sort_values()
plt.figure(figsize=(9,6)); imp.plot(kind="barh")
plt.title("Top features in exploratory model"); plt.xlabel("Random forest importance")
plt.tight_layout(); plt.show()

## 10. Robustness: remove one modality at a time

In [ ]:
robust=[]
for dropped in ["expression","methylation","clinical"]:
    blocks=[]
    if dropped!="expression": blocks.append(block(E).add_prefix("EXP_"))
    if dropped!="methylation": blocks.append(block(METH).add_prefix("METH_"))
    if dropped!="clinical": blocks.append(block(C).add_prefix("CLIN_"))
    Xalt=pd.concat(blocks,axis=1)
    lab=KMeans(n_clusters=3,n_init=20,random_state=SEED).fit_predict(Xalt)
    robust.append({"analysis":f"Without {dropped}",
                   "silhouette":silhouette_score(Xalt,lab),
                   "ARI_vs_synthetic_truth":adjusted_rand_score(truth,lab)})
robustness=pd.DataFrame(robust)
display(robustness)

## 11. Export results and summary

In [ ]:
sample_results=pd.DataFrame({
 "sample_id":meta.sample_id,
 "discovered_cluster":labels["Integrated"],
 "simulation_truth_for_validation_only":truth,
 "injected_discordance_for_validation_only":meta.injected_discordance
}).merge(anom,on="sample_id",how="left")
sample_results.to_csv(f"{OUT}/sample_level_results.csv",index=False)
cluster_metrics.to_csv(f"{OUT}/clustering_metrics.csv",index=False)
robustness.to_csv(f"{OUT}/modality_robustness.csv",index=False)
imp.rename("importance").to_csv(f"{OUT}/feature_importance.csv")
summary={
 "project":"OMNISIGNAL","data":"synthetic demonstration cohort",
 "samples":int(n),"expression_features":int(ng),"methylation_features":int(nm),
 "best_ARI_representation":str(cluster_metrics.iloc[0].representation),
 "best_ARI":float(cluster_metrics.iloc[0].ARI_vs_synthetic_truth),
 "caution":"No real biological or clinical conclusions can be drawn from synthetic data."
}
with open(f"{OUT}/project_summary.json","w") as f: json.dump(summary,f,indent=2)
print(json.dumps(summary,indent=2))
print("Saved exports to",OUT)

## 12. Research interpretation and next steps

Use these questions when analysing results:
- Does integrated clustering recover hidden groups better than individual modalities?
- Which modality most often drives discordance flags?
- Are cluster signatures biologically interpretable?
- Does removing one modality change the result substantially?
- Are findings stable across random seeds and preprocessing strategies?

### Limitations
- All observations are synthetic.
- Equal block scaling is a baseline, not a universal integration method.
- Real omics studies must account for batch effects, confounders, tissue type and platform differences.
- Independent cohort validation and pathway enrichment are needed before biological interpretation.
- No output is suitable for diagnosis or treatment decisions.

### Real-data extension
Select a public cohort with matched expression, methylation and clinical data. Verify sample identifiers, platform compatibility, access conditions and outcome definitions before integration. Keep published labels separate from unsupervised discovery and validate on an independent cohort where possible.

**Portfolio description:** *OMNISIGNAL is an explainable multi-omics discovery engine that integrates molecular and clinical data to uncover hidden subtypes and identify cross-modal biological discordance.*